In [1]:
import copy
import math

import numpy as np

In [2]:
MEASUREMENTS = {
    "base_to_leg_front": 0.075,
    "base_to_leg_back": 0.075,
    "base_to_leg_left": 0.0445,
    "base_to_leg_right": 0.0335,
    "trans_1_2": [0, 0, -0.039],
    "trans_2_3": [0, -0.0494, 0.0685],
    "trans_3_ee": [0.06231, -0.06216, -0.018],
}

In [3]:
LEG_IDS = ["leg_front_l", "leg_back_l", "leg_front_r", "leg_back_r"]

In [4]:
thetas = {lid: [math.radians(30), math.radians(45), math.radians(60)] for lid in LEG_IDS}

In [5]:
def forward_kinematics(
    thetas: dict[str, list[float]], measurements: dict[str, float | list[float]]
) -> dict[str, list[float]]:
    """
    thetas: dict of leg_id to joint angles (_1, _2, _3) for the leg.
    """

    def rotation_x(angle):
        # rotation about the x-axis implemented for you
        return np.array(
            [
                [1, 0, 0, 0],
                [0, np.cos(angle), -np.sin(angle), 0],
                [0, np.sin(angle), np.cos(angle), 0],
                [0, 0, 0, 1],
            ]
        )

    def rotation_y(angle):
        return np.array(
            [
                [np.cos(angle), 0, np.sin(angle), 0],
                [0, 1, 0, 0],
                [-np.sin(angle), 0, np.cos(angle), 0],
                [0, 0, 0, 1],
            ]
        )

    def rotation_z(angle):
        return np.array(
            [
                [np.cos(angle), -np.sin(angle), 0, 0],
                [np.sin(angle), np.cos(angle), 0, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1],
            ]
        )

    def translation(x, y, z):
        return np.array(
            [
                [1, 0, 0, x],
                [0, 1, 0, y],
                [0, 0, 1, z],
                [0, 0, 0, 1],
            ]
        )

    # theta is positive when the Z-axis points out of the BOTTOM (i.e. uncovered metal parts) of the BLDC motor.
    # Since we keep the frame orientation the same on both left and right sides, motors 1 and 3 will use negative angles on the left and positive angles on the right.
    out: dict[str, list[float]] = {}

    base_to_leg_front = measurements["base_to_leg_front"]
    base_to_leg_back = measurements["base_to_leg_back"]

    base_to_leg_left = measurements["base_to_leg_left"]
    base_to_leg_right = measurements["base_to_leg_right"]

    trans_1_2 = translation(*measurements["trans_1_2"])
    trans_2_3 = translation(*measurements["trans_2_3"])
    trans_3_ee = translation(*measurements["trans_3_ee"])

    rot_0_1 = rotation_x(1.57080)
    rot_1_2 = rotation_y(-1.57080)
    rot_2_3 = rotation_y(1.57080)

    front_l = thetas["leg_front_l"]
    T_0_1 = translation(base_to_leg_front, base_to_leg_left, 0) @ rot_0_1 @ rotation_z(-front_l[0])
    T_1_2 = trans_1_2 @ rot_1_2 @ rotation_z(+front_l[1])
    T_2_3 = trans_2_3 @ rot_2_3 @ rotation_z(-front_l[2])
    T_3_ee = trans_3_ee
    T_0_ee = T_0_1 @ T_1_2 @ T_2_3 @ T_3_ee
    out["leg_front_l"] = T_0_ee[:3, 3].copy()

    back_l = thetas["leg_back_l"]
    T_0_1 = translation(-base_to_leg_back, base_to_leg_left, 0) @ rot_0_1 @ rotation_z(-back_l[0])
    T_1_2 = trans_1_2 @ rot_1_2 @ rotation_z(+back_l[1])
    T_2_3 = trans_2_3 @ rot_2_3 @ rotation_z(-back_l[2])
    T_3_ee = trans_3_ee
    T_0_ee = T_0_1 @ T_1_2 @ T_2_3 @ T_3_ee
    out["leg_back_l"] = T_0_ee[:3, 3].copy()

    front_r = thetas["leg_front_r"]
    # base_link frame is FLU, so points to the right are negative
    T_0_1 = (
        translation(base_to_leg_front, -base_to_leg_right, 0) @ rot_0_1 @ rotation_z(+front_r[0])
    )
    T_1_2 = trans_1_2 @ rot_1_2 @ rotation_z(+front_r[1])
    T_2_3 = trans_2_3 @ rot_2_3 @ rotation_z(+front_r[2])
    T_3_ee = trans_3_ee
    T_0_ee = T_0_1 @ T_1_2 @ T_2_3 @ T_3_ee
    out["leg_front_r"] = T_0_ee[:3, 3].copy()

    back_r = thetas["leg_back_r"]
    T_0_1 = (
        translation(-base_to_leg_front, -base_to_leg_right, 0) @ rot_0_1 @ rotation_z(+back_r[0])
    )
    T_1_2 = trans_1_2 @ rot_1_2 @ rotation_z(+back_r[1])
    T_2_3 = trans_2_3 @ rot_2_3 @ rotation_z(+back_r[2])
    T_3_ee = trans_3_ee
    T_0_ee = T_0_1 @ T_1_2 @ T_2_3 @ T_3_ee
    out["leg_back_r"] = T_0_ee[:3, 3].copy()

    return out

In [6]:
forward_kinematics(thetas, MEASUREMENTS)

{'leg_front_l': array([-0.05785841,  0.00116349, -0.04776266]),
 'leg_back_l': array([-0.20785841,  0.00116349, -0.04776266]),
 'leg_front_r': array([ 0.10501779, -0.00052288, -0.01901811]),
 'leg_back_r': array([-0.04498221, -0.00052288, -0.01901811])}

In [8]:
simulated_inputs = {
    "theta_degs": [0.0, 20.0, 45.0, 70.0],
    "l1_l2_err_cm": [0.2, 0.4, 0.8, 1.0],
}

In [9]:
# Compute errors for front left leg
def simulate_errors(inputs: dict[str, list[float]]) -> dict[str, list[float]]:
    """Return error in EE position for each dimension"""
    out = {}
    measurements = copy.deepcopy(MEASUREMENTS)
    for deg in inputs["theta_degs"]:
        thetas = {lid: [math.radians(deg) for _ in range(3)] for lid in LEG_IDS}
        ref_ee = forward_kinematics(thetas, MEASUREMENTS)
        for err_cm in inputs["l1_l2_err_cm"]:
            measurements["trans_1_2"][2] = measurements["trans_1_2"][2] + err_cm / 100.0
            ee = forward_kinematics(thetas, measurements)

            diff = ee["leg_front_l"] - ref_ee["leg_front_l"]
            out[(deg, err_cm)] = np.round(diff, 2)

    return out

In [10]:
errs = simulate_errors(simulated_inputs)

In [11]:
errs

{(0.0, 0.2): array([ 0., -0., -0.]),
 (0.0, 0.4): array([ 0.  , -0.01, -0.  ]),
 (0.0, 0.8): array([ 0.  , -0.01, -0.  ]),
 (0.0, 1.0): array([ 0.  , -0.02, -0.  ]),
 (20.0, 0.2): array([ 0.  , -0.03, -0.  ]),
 (20.0, 0.4): array([ 0.  , -0.03, -0.  ]),
 (20.0, 0.8): array([ 0.  , -0.04, -0.  ]),
 (20.0, 1.0): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.2): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.4): array([ 0.  , -0.05, -0.  ]),
 (45.0, 0.8): array([ 0.  , -0.06, -0.  ]),
 (45.0, 1.0): array([ 0.  , -0.07, -0.  ]),
 (70.0, 0.2): array([ 0.  , -0.07, -0.  ]),
 (70.0, 0.4): array([ 0.  , -0.08, -0.  ]),
 (70.0, 0.8): array([ 0.  , -0.09, -0.  ]),
 (70.0, 1.0): array([ 0. , -0.1, -0. ])}